In [ ]:
# Download pretrained model
from IPython.display import HTML, clear_output, Audio
import os
from base64 import b64decode, b64encode
import io
import ffmpeg
import shutil
import subprocess
import librosa
import soundfile as sf
import moviepy.editor as mp
from pydub import AudioSegment
from scipy.io.wavfile import read as wav_read
from google.colab import files
from IPython.display import display
import speech_recognition as sr
from datetime import timedelta
from google.colab.output import eval_js

!rm -rf /content/sample_data
!mkdir /content/sample_data
!git clone https://github.com/justinjohn0306/Wav2Lip
!wget 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip.pth' -O '/content/Wav2Lip/checkpoints/wav2lip.pth'
!wget 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth' -O '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
!wget 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/resnet50.pth' -O '/content/Wav2Lip/checkpoints/resnet50.pth'
!wget 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/mobilenet.pth' -O '/content/Wav2Lip/checkpoints/mobilenet.pth'
!pip install https://raw.githubusercontent.com/AwaleSajil/ghc/master/ghc-1.0-py3-none-any.whl git+https://github.com/elliottzheng/batch-face.git@master ffmpeg-python mediapipe pydub moviepy
!pip install --upgrade googletrans==4.0.0-rc1 google-cloud-texttospeech python-docx gTTS pydub pytube pyttsx3 SpeechRecognition PyDictionary textblob translate==3.5.0 gcp-texttospeech==1.0.1

clear_output()
print("All set and ready!")

def get_audio():
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])

    process = (ffmpeg.input('pipe:0').output('pipe:1', format='wav').run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True))
    output, err = process.communicate(input=binary)

    riff_chunk_size = len(output) - 8
    q = riff_chunk_size
    b = []
    for i in range(4):
        q, r = divmod(q, 256)
        b.append(r)
    riff = output[:4] + bytes(b) + output[8:]

    sr, audio = wav_read(io.BytesIO(riff))
    return audio, sr

def show_video(path):
    mp4 = open(str(path), 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=700 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """)

def mp3_to_wav(mp3_file, wav_file):
    audio = AudioSegment.from_mp3(mp3_file)
    audio.export(wav_file, format="wav")

mp3_file = "/content/telugu_audio.mp3"
wav_file = "/content/TryAudio.wav"
mp3_to_wav(mp3_file, wav_file)

def display_audio(audio_path):
    display(Audio(audio_path))

PATH_TO_YOUR_AUDIO = "/content/TryAudio.wav"
audio, sr = librosa.load(PATH_TO_YOUR_AUDIO, sr=None)
sf.write('/content/sample_data/input_audio.wav', audio, sr, format='wav')

clear_output()
display_audio('/content/sample_data/input_audio.wav')

def resize_video(video_path, new_resolution):
    import cv2
    video = cv2.VideoCapture(video_path)
    fourcc = int(video.get(cv2.CAP_PROP_FOURCC))
    fps = video.get(cv2.CAP_PROP_FPS)
    width, height = new_resolution
    output_path = os.path.splitext(video_path)[0] + '_720p.mp4'
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    while True:
        success, frame = video.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, new_resolution)
        writer.write(resized_frame)
    video.release()
    writer.release()

PATH_TO_YOUR_VIDEO = "/content/TryVideo.mp4"
video_resolution = mp.VideoFileClip(PATH_TO_YOUR_VIDEO).size

if video_resolution[0] >= 1920 or video_resolution[1] >= 1080:
    !ffmpeg -i {PATH_TO_YOUR_VIDEO} -vf scale=1280:720 /content/sample_data/input_vid.mp4
    PATH_TO_YOUR_VIDEO = '/content/sample_data/input_vid.mp4'

clear_output()
print("Input Video")
show_video(PATH_TO_YOUR_VIDEO)

# Generate Video with Wav2Lip
output_file_path = '/content/Wav2Lip/results/result_voice.mp4'
if os.path.exists(output_file_path):
    os.remove(output_file_path)

pad_top, pad_bottom, pad_left, pad_right = 0, 10, 0, 0
rescaleFactor = 1
nosmooth = True
use_hd_model = False
checkpoint_path = '/content/Wav2Lip/checkpoints/wav2lip.pth' if not use_hd_model else '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'

%cd /content/Wav2Lip
!python inference.py --checkpoint_path {checkpoint_path} --face "/content/sample_data/input_vid.mp4" --audio "/content/sample_data/input_audio.wav" --pads {pad_top} {pad_bottom} {pad_left} {pad_right} --resize_factor {rescaleFactor} {'--nosmooth' if nosmooth else ''}

if os.path.exists(output_file_path):
    clear_output()
    print("Final Video Preview")
    print("Download this video from", output_file_path)
    show_video(output_file_path)
else:
    print("Processing failed. Output video not found.")

def recognize_text(audio_path):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = recognizer.record(source)
    try:
        text = recognizer.recognize_google(audio_data, language="hi-IN")
        return text
    except sr.UnknownValueError:
        print("Google Speech Recognition could not understand the audio.")
        return ""
    except sr.RequestError as e:
        print(f"Could not request results from Google Speech Recognition service; {e}")
        return ""

def generate_srt_from_audio(audio_path, chunk_duration_ms=5000):
    audio = AudioSegment.from_file(audio_path)
    total_duration = len(audio)
    srt_content = ""
    c = 1
    for i in range(0, total_duration, chunk_duration_ms):
        chunk_start = i
        chunk_end = min(i + chunk_duration_ms, total_duration)
        audio_chunk = audio[chunk_start:chunk_end]
        chunk_audio_path = f"chunk_{i}.wav"
        audio_chunk.export(chunk_audio_path, format="wav")
        text = recognize_text(chunk_audio_path)
        os.remove(chunk_audio_path)
        start_time = timedelta(milliseconds=chunk_start)
        end_time = timedelta(milliseconds=chunk_end)
        srt_content += f"{c}\n{start_time},000 --> {end_time},000\n{text}\n\n"
        c += 1
    return srt_content

audio_file_path = "/content/test.wav"
recognized_text = recognize_text(audio_file_path)
srt_text = generate_srt_from_audio(audio_file_path)

if recognized_text:
    srt_file_path = "/content/output.srt"
    with open(srt_file_path, "w") as srt_file:
        srt_file.write(srt_text)
    print("SRT file generated successfully.")
else:
    print("No recognized text to generate SRT file.")

def burn_subtitles_into_video(video_file, subtitle_file, output_file, font_size=16, font_style='bold'):
    command = [
        'ffmpeg',
        '-i', video_file,
        '-vf', f"subtitles={subtitle_file}:force_style='Fontname=NotoSans-Regular.ttf,"
               f"Fontsize={font_size},PrimaryColour=&HFFFFFF&,BackColour=&H80000000&,OutlineColour=&H000000&,Outline=2,Style={font_style}'",
        '-c:a', 'copy',
        '-scodec', 'ass',
        output_file
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print("Error burning subtitles into the video:")
        print(result.stderr)
        return False
    return True

video_path = "/content/video_output.mp4"
subtitle_path = "/content/output.srt"
output_path = "/content/final_output.mp4"

if shutil.which('ffmpeg') is None:
    print("ffmpeg is not installed or not in the system's PATH.")
else:
    if burn_subtitles_into_video(video_path, subtitle_path, output_path):
        print("Subtitles have been burned into the video!")
    else:
        print("Failed to burn subtitles into the video.")
